# Causal DAG — From Correlation to Causation

**RetentionIQ** | Notebook 03 of 07
*Prescriptive retention system for 250+ location fitness franchise (Brazil)*

> Previous: [02_survival_analysis.ipynb](./02_survival_analysis.ipynb) — member tenure modeling with Kaplan-Meier, Cox PH, and AFT
> Next: [04_treatment_effects.ipynb](./04_treatment_effects.ipynb) — heterogeneous treatment effect estimation via CausalForestDML
> Architecture decisions: [docs/ARCHITECTURE.md](../docs/ARCHITECTURE.md) (see ADR-003: Separate models for Regular vs Aggregator)
> Causal config: [configs/causal.yaml](../configs/causal.yaml)

## 1. Why Causal Inference Matters Here

Most ML projects stop at prediction. We could build an XGBoost model that predicts churn with 0.88 AUC and call it a day. But here's the problem: **prediction doesn't tell you what to DO**. A model that predicts "this member will churn" is useless unless we know which intervention prevents it — and that's a causal question, not a predictive one.

Worse, our retention action data has built-in **selection bias**. Managers target at-risk members, which means action recipients have HIGHER churn rates. A naive model would learn that "retention actions increase churn." The only way to get the right answer is to model the data-generating process explicitly — and that starts with a causal DAG.

**The stakes are real.** With 250+ locations and a monthly retention budget, misallocating spend because we confused correlation with causation costs real money. If we naively target members who "look like" past churners with retention actions, we'll waste budget on members who would have churned regardless (no treatment effect) and miss members who would respond to an SMS nudge but don't fit the "typical churner" profile.

This notebook defines the causal graph — the foundation for everything in notebooks 04 (estimation) and 05 (optimization).

## 2. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from pathlib import Path

import dowhy
from dowhy import CausalModel

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 8),
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.facecolor": "white",
})

OBSERVATION_END = pd.Timestamp("2026-03-01")
DATA_DIR = Path("../data/raw/")

print(f"Observation window ends: {OBSERVATION_END.date()}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"DoWhy version: {dowhy.__version__}")

In [ ]:
# ── Load raw parquet data ──────────────────────────────────────────────────────
members = pd.read_parquet(DATA_DIR / "members.parquet")
visits = pd.read_parquet(DATA_DIR / "visits.parquet")
retention_actions = pd.read_parquet(DATA_DIR / "retention_actions.parquet")

# Parse dates
members["join_date"] = pd.to_datetime(members["join_date"])
members["cancel_date"] = pd.to_datetime(members["cancel_date"])
visits["visit_date"] = pd.to_datetime(visits["visit_date"])
retention_actions["action_date"] = pd.to_datetime(retention_actions["action_date"])

print(f"Members:           {len(members):>10,}")
print(f"  Churned:         {members['churned'].sum():>10,} ({members['churned'].mean():.1%})")
print(f"  Regular:         {(members['contract_source'] == 'regular').sum():>10,}")
print(f"  Aggregator:      {(members['contract_source'] == 'aggregator').sum():>10,}")
print(f"Visits:            {len(visits):>10,}")
print(f"Retention actions: {len(retention_actions):>10,}")
print(f"\nAction types:\n{retention_actions['action_type'].value_counts().to_string()}")

## 3. Defining the Causal Graph

A causal DAG encodes our assumptions about what causes what. Every edge is a claim: "X directly affects Y." Every **missing** edge is also a claim: "X does **not** directly affect Y." This is uncomfortable for many ML practitioners — we're making assumptions explicit instead of hiding them in a black box. But that's the point: **you can critique assumptions you can see.**

Our DAG encodes the following domain knowledge (informed by operating a 250+ unit franchise network):

| Edge | Domain justification |
|------|---------------------|
| `visit_frequency` -> `retention_action_taken` | Managers notice declining visits and intervene |
| `visit_frequency` -> `churned_30d` | Fewer visits is a strong leading indicator of churn |
| `tenure_months` -> `retention_action_taken` | Newer members are less likely to get proactive retention (they're "too new to worry about") |
| `tenure_months` -> `churned_30d` | Tenure has a non-monotonic relationship with churn (high early, dips, rises at contract renewal) |
| `plan_price` -> `churned_30d` | Higher-price members churn more (price sensitivity) but also self-select into commitment |
| `location_churn_rate` -> `retention_action_taken` | High-churn locations trigger more proactive actions (manager pressure from regional ops) |
| `location_churn_rate` -> `churned_30d` | Location-level factors (facility quality, staff, neighborhood) affect member retention |
| `enrollment_month` -> `churned_30d` | January cohorts churn faster (New Year's resolution effect) |
| `payment_delay_flag` -> `retention_action_taken` | Payment delays trigger automated retention workflows |
| `payment_delay_flag` -> `churned_30d` | Payment friction is both a symptom and cause of disengagement |
| `retention_action_taken` -> `churned_30d` | **THE CAUSAL EFFECT WE WANT TO ESTIMATE** |

Note what is **not** in the DAG: `age`, `gender`, and `plan_type` are not included as confounders because, in our domain, they do not directly cause managers to assign retention actions (managers don't have demographic dashboards). They may affect churn, but they're not on the backdoor path. Including non-confounders as controls can actually **increase** variance without reducing bias — a common mistake.

In [ ]:
# ── Build the causal DAG ───────────────────────────────────────────────────────

dag = nx.DiGraph()

# Define all edges with domain justification
causal_edges = [
    # Confounders → Treatment (selection bias paths)
    ("visit_frequency", "retention_action_taken"),
    ("tenure_months", "retention_action_taken"),
    ("location_churn_rate", "retention_action_taken"),
    ("payment_delay_flag", "retention_action_taken"),

    # Confounders → Outcome (direct effects on churn)
    ("visit_frequency", "churned_30d"),
    ("tenure_months", "churned_30d"),
    ("plan_price", "churned_30d"),
    ("location_churn_rate", "churned_30d"),
    ("enrollment_month", "churned_30d"),
    ("payment_delay_flag", "churned_30d"),

    # Treatment → Outcome (THE CAUSAL EFFECT)
    ("retention_action_taken", "churned_30d"),
]

dag.add_edges_from(causal_edges)

# Verify DAG is acyclic (a causal graph with cycles is incoherent)
assert nx.is_directed_acyclic_graph(dag), "Graph has cycles — not a valid causal DAG!"
print(f"DAG nodes: {dag.number_of_nodes()}")
print(f"DAG edges: {dag.number_of_edges()}")
print(f"Is a valid DAG: {nx.is_directed_acyclic_graph(dag)}")

In [ ]:
# ── Visualize the DAG with role-based color coding ─────────────────────────────

# Node roles
treatment_nodes = {"retention_action_taken"}
outcome_nodes = {"churned_30d"}
confounder_nodes = {
    "visit_frequency", "tenure_months", "plan_price",
    "location_churn_rate", "enrollment_month", "payment_delay_flag",
}

def get_node_color(node: str) -> str:
    if node in treatment_nodes:
        return "#3498db"   # blue
    elif node in outcome_nodes:
        return "#e74c3c"   # red
    else:
        return "#95a5a6"   # gray (confounders)

node_colors = [get_node_color(n) for n in dag.nodes()]

# Manual layout for readability — confounders on left, treatment center, outcome right
pos = {
    # Confounders (left column, spread vertically)
    "visit_frequency":      (-2.5,  2.0),
    "tenure_months":        (-2.5,  1.0),
    "plan_price":           (-2.5,  0.0),
    "location_churn_rate":  (-2.5, -1.0),
    "enrollment_month":     (-2.5, -2.0),
    "payment_delay_flag":   (-2.5, -3.0),
    # Treatment (center)
    "retention_action_taken": (0.0, 0.0),
    # Outcome (right)
    "churned_30d":           (2.5, 0.0),
}

# Readable labels
labels = {
    "visit_frequency": "Visit\nFrequency",
    "tenure_months": "Tenure\n(months)",
    "plan_price": "Plan\nPrice",
    "location_churn_rate": "Location\nChurn Rate",
    "enrollment_month": "Enrollment\nMonth",
    "payment_delay_flag": "Payment\nDelay Flag",
    "retention_action_taken": "Retention\nAction\n(TREATMENT)",
    "churned_30d": "Churned\n30d\n(OUTCOME)",
}

fig, ax = plt.subplots(figsize=(16, 10))

# Draw edges with curved arrows for clarity
edge_colors = []
edge_widths = []
edge_styles = []
for u, v in dag.edges():
    if u == "retention_action_taken" and v == "churned_30d":
        edge_colors.append("#e74c3c")
        edge_widths.append(3.0)
    elif v == "retention_action_taken":
        edge_colors.append("#3498db")
        edge_widths.append(1.5)
    else:
        edge_colors.append("#7f8c8d")
        edge_widths.append(1.2)

nx.draw_networkx_edges(
    dag, pos, ax=ax,
    edge_color=edge_colors,
    width=edge_widths,
    arrows=True,
    arrowsize=20,
    arrowstyle="-|>",
    connectionstyle="arc3,rad=0.1",
    min_source_margin=30,
    min_target_margin=30,
)

nx.draw_networkx_nodes(
    dag, pos, ax=ax,
    node_color=node_colors,
    node_size=3000,
    edgecolors="white",
    linewidths=2,
    alpha=0.9,
)

nx.draw_networkx_labels(
    dag, pos, labels=labels, ax=ax,
    font_size=9,
    font_weight="bold",
    font_color="white",
)

# Legend
legend_elements = [
    mpatches.Patch(facecolor="#3498db", label="Treatment"),
    mpatches.Patch(facecolor="#e74c3c", label="Outcome"),
    mpatches.Patch(facecolor="#95a5a6", label="Confounder"),
    Line2D([0], [0], color="#e74c3c", linewidth=3, label="Causal effect of interest"),
    Line2D([0], [0], color="#3498db", linewidth=1.5, label="Selection bias path"),
    Line2D([0], [0], color="#7f8c8d", linewidth=1.2, label="Direct effect on outcome"),
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=10, framealpha=0.9)

ax.set_title(
    "Causal DAG: Retention Actions and Member Churn\n"
    "Each edge represents a claimed direct causal relationship",
    fontsize=14, fontweight="bold", pad=20,
)
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. The Backdoor Criterion

To estimate the causal effect of retention actions on churn, we need to "block all backdoor paths" — paths that create spurious association through common causes. The **backdoor criterion** (Pearl, 2009) tells us which variables we need to condition on.

In our DAG, the confounders that create backdoor paths are:
- **visit_frequency** — managers see declining visits and intervene; declining visits also predict churn
- **tenure_months** — newer members are less likely to receive actions; tenure also predicts churn
- **plan_price** — affects churn directly (only reaches treatment through other confounders, but we include it for safety)
- **location_churn_rate** — high-churn locations trigger more actions; location quality affects churn
- **enrollment_month** — seasonality affects both manager behavior and churn patterns
- **payment_delay_flag** — payment delays trigger automated retention workflows and predict churn

If we miss a confounder (unmeasured confounding), our estimate is biased. If we condition on a **collider** or **mediator**, we introduce new bias. Getting the adjustment set right is the whole ballgame.

**A common mistake:** Including `plan_type` as a confounder. In our setting, plan type is a **pre-treatment variable** that affects churn but does NOT affect treatment assignment (managers don't filter by plan type when deciding whom to call). Including it is harmless for bias but increases variance. We keep our adjustment set minimal and justified.

In [ ]:
# ── Prepare data for DoWhy identification ──────────────────────────────────────
# We need a flat DataFrame with treatment, outcome, and all confounders.
# This mirrors the data prep in src/causal/dag.py but we do it here for
# transparency — no hidden transformations.

# Compute derived features
members["tenure_months"] = (
    (members["cancel_date"].fillna(OBSERVATION_END) - members["join_date"]).dt.days / 30
).clip(lower=0)

members["enrollment_month"] = members["join_date"].dt.month

# Visit frequency in last 30 days of membership (or observation end)
reference_date = members["cancel_date"].fillna(OBSERVATION_END)
visit_counts = (
    visits.merge(members[["member_id", "cancel_date"]], on="member_id", how="left")
    .assign(
        ref_date=lambda df: df["cancel_date"].fillna(OBSERVATION_END),
        days_before_ref=lambda df: (df["ref_date"] - df["visit_date"]).dt.days,
    )
    .query("0 <= days_before_ref <= 30")
    .groupby("member_id")
    .size()
    .rename("visit_frequency")
)
members = members.merge(visit_counts, on="member_id", how="left")
members["visit_frequency"] = members["visit_frequency"].fillna(0).astype(int)

# Location churn rate
loc_churn = (
    members.groupby("location_id")["churned"]
    .mean()
    .rename("location_churn_rate")
)
members = members.merge(loc_churn, on="location_id", how="left")

# Payment delay flag (synthetic: correlate with churn + low visits)
# In production this comes from transaction data; here we simulate it
rng = np.random.default_rng(42)
payment_delay_prob = (
    0.05
    + 0.15 * members["churned"].astype(float)
    + 0.10 * (members["visit_frequency"] < 2).astype(float)
).clip(0, 1)
members["payment_delay_flag"] = rng.binomial(1, payment_delay_prob).astype(int)

# Retention action indicator
action_members = retention_actions["member_id"].unique()
members["retention_action_taken"] = members["member_id"].isin(action_members).astype(int)

# 30-day churn outcome
members["churned_30d"] = members["churned"].astype(int)

print("Analysis DataFrame ready:")
print(f"  Shape: {members.shape}")
print(f"  Treatment rate: {members['retention_action_taken'].mean():.1%}")
print(f"  Outcome rate:   {members['churned_30d'].mean():.1%}")
print(f"\nNaive comparison (WRONG — ignores confounding):")
print(f"  Churn rate WITH action:    {members.loc[members['retention_action_taken']==1, 'churned_30d'].mean():.1%}")
print(f"  Churn rate WITHOUT action: {members.loc[members['retention_action_taken']==0, 'churned_30d'].mean():.1%}")
print(f"\n  ^ This makes it look like actions INCREASE churn — classic selection bias!")

In [ ]:
# ── Use DoWhy to identify the causal estimand ─────────────────────────────────
# DoWhy formalizes the identification step: given the DAG, what statistical
# quantity answers our causal question?

# Build the graph string in DOT format for DoWhy
confounders = [
    "visit_frequency", "tenure_months", "plan_price",
    "location_churn_rate", "enrollment_month", "payment_delay_flag",
]

edges_str = []
for c in confounders:
    edges_str.append(f'"{c}" -> "retention_action_taken"')
    edges_str.append(f'"{c}" -> "churned_30d"')
edges_str.append('"retention_action_taken" -> "churned_30d"')

graph_str = "digraph { " + "; ".join(edges_str) + " }"

# Subsample for DoWhy (full dataset is too large for interactive use)
sample_size = min(50_000, len(members))
members_sample = members.sample(n=sample_size, random_state=42)

model = CausalModel(
    data=members_sample,
    treatment="retention_action_taken",
    outcome="churned_30d",
    graph=graph_str,
)

# Identify the estimand
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print("=" * 80)
print("IDENTIFIED ESTIMAND")
print("=" * 80)
print(identified_estimand)
print("\nBackdoor variables (adjustment set):")
backdoor_vars = identified_estimand.get_backdoor_variables()
for var in backdoor_vars:
    print(f"  - {var}")

## 5. Why Separate DAGs for Regular vs Aggregator

The causal structure is **DIFFERENT** for these two segments. For regular members, the gym can offer discounts and plan upgrades — these are valid treatments. For aggregator members (Gympass, TotalPass, Wellhub), pricing is controlled by the platform. Including discount offers or plan upgrades as treatments for aggregator members would **violate the DAG** — these aren't interventions the gym can perform.

This isn't a modeling convenience — it's a **causal necessity**. The wrong DAG gives the wrong causal effect. This decision is documented in [ADR-003](../docs/ARCHITECTURE.md) and implemented in the segment-specific configs in [configs/causal.yaml](../configs/causal.yaml).

**Practical consequence:** When the budget optimizer (notebook 05) allocates retention spend, it must respect these constraints. Recommending "offer a 20% discount" to a Gympass member is not just ineffective — it's operationally impossible.

In [ ]:
# ── Side-by-side DAGs: Regular vs Aggregator ──────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# ── Regular member DAG (more treatment options) ──────────────────────────────
dag_regular = nx.DiGraph()
regular_edges = [
    ("visit_frequency", "retention_action_taken"),
    ("tenure_months", "retention_action_taken"),
    ("location_churn_rate", "retention_action_taken"),
    ("payment_delay_flag", "retention_action_taken"),
    ("visit_frequency", "churned_30d"),
    ("tenure_months", "churned_30d"),
    ("plan_price", "churned_30d"),
    ("location_churn_rate", "churned_30d"),
    ("enrollment_month", "churned_30d"),
    ("payment_delay_flag", "churned_30d"),
    ("retention_action_taken", "churned_30d"),
    # Regular-specific: discount and upgrade paths
    ("discount_received", "churned_30d"),
    ("plan_upgrade", "churned_30d"),
    ("upgrade_history", "retention_action_taken"),
    ("upgrade_history", "churned_30d"),
]
dag_regular.add_edges_from(regular_edges)

pos_regular = {
    "visit_frequency":        (-2.5,  2.5),
    "tenure_months":          (-2.5,  1.5),
    "plan_price":             (-2.5,  0.5),
    "location_churn_rate":    (-2.5, -0.5),
    "enrollment_month":       (-2.5, -1.5),
    "payment_delay_flag":     (-2.5, -2.5),
    "upgrade_history":        (-2.5, -3.5),
    "retention_action_taken": ( 0.0,  0.0),
    "discount_received":      ( 0.0, -2.0),
    "plan_upgrade":           ( 0.0, -3.0),
    "churned_30d":            ( 2.5,  0.0),
}

regular_colors = []
for n in dag_regular.nodes():
    if n == "retention_action_taken":
        regular_colors.append("#3498db")
    elif n == "churned_30d":
        regular_colors.append("#e74c3c")
    elif n in ("discount_received", "plan_upgrade"):
        regular_colors.append("#2ecc71")  # green for regular-specific treatments
    else:
        regular_colors.append("#95a5a6")

labels_regular = {n: n.replace("_", "\n") for n in dag_regular.nodes()}

nx.draw(
    dag_regular, pos_regular, ax=axes[0],
    node_color=regular_colors,
    node_size=2000,
    with_labels=True,
    labels=labels_regular,
    font_size=7,
    font_weight="bold",
    font_color="white",
    edge_color="#7f8c8d",
    arrows=True,
    arrowsize=15,
    connectionstyle="arc3,rad=0.08",
    edgecolors="white",
    linewidths=1.5,
)
axes[0].set_title(
    "Regular Members\n(discount + upgrade are valid treatments)",
    fontsize=13, fontweight="bold", pad=15,
)

# ── Aggregator member DAG (restricted treatments) ────────────────────────────
dag_agg = nx.DiGraph()
agg_edges = [
    ("visit_frequency", "retention_action_taken"),
    ("tenure_months", "retention_action_taken"),
    ("location_churn_rate", "retention_action_taken"),
    ("payment_delay_flag", "retention_action_taken"),
    ("visit_frequency", "churned_30d"),
    ("tenure_months", "churned_30d"),
    ("plan_price", "churned_30d"),
    ("location_churn_rate", "churned_30d"),
    ("enrollment_month", "churned_30d"),
    ("payment_delay_flag", "churned_30d"),
    ("retention_action_taken", "churned_30d"),
    # Aggregator-specific confounder
    ("aggregator_platform", "churned_30d"),
    ("aggregator_platform", "retention_action_taken"),
]
dag_agg.add_edges_from(agg_edges)

pos_agg = {
    "visit_frequency":        (-2.5,  2.5),
    "tenure_months":          (-2.5,  1.5),
    "plan_price":             (-2.5,  0.5),
    "location_churn_rate":    (-2.5, -0.5),
    "enrollment_month":       (-2.5, -1.5),
    "payment_delay_flag":     (-2.5, -2.5),
    "aggregator_platform":    (-2.5, -3.5),
    "retention_action_taken": ( 0.0,  0.0),
    "churned_30d":            ( 2.5,  0.0),
}

agg_colors = []
for n in dag_agg.nodes():
    if n == "retention_action_taken":
        agg_colors.append("#3498db")
    elif n == "churned_30d":
        agg_colors.append("#e74c3c")
    elif n == "aggregator_platform":
        agg_colors.append("#f39c12")  # orange for aggregator-specific
    else:
        agg_colors.append("#95a5a6")

labels_agg = {n: n.replace("_", "\n") for n in dag_agg.nodes()}

nx.draw(
    dag_agg, pos_agg, ax=axes[1],
    node_color=agg_colors,
    node_size=2000,
    with_labels=True,
    labels=labels_agg,
    font_size=7,
    font_weight="bold",
    font_color="white",
    edge_color="#7f8c8d",
    arrows=True,
    arrowsize=15,
    connectionstyle="arc3,rad=0.08",
    edgecolors="white",
    linewidths=1.5,
)
axes[1].set_title(
    "Aggregator Members\n(NO discount/upgrade — platform controls pricing)",
    fontsize=13, fontweight="bold", pad=15,
)

# Add annotation about structural difference
fig.text(
    0.5, 0.02,
    "Key structural difference: Regular members have discount_received and plan_upgrade as valid interventions.\n"
    "Aggregator members have aggregator_platform as an additional confounder, with no pricing levers available to the gym.",
    ha="center", fontsize=11, style="italic",
    bbox=dict(boxstyle="round,pad=0.5", facecolor="#ecf0f1", alpha=0.8),
)

# Legend
legend_elements = [
    mpatches.Patch(facecolor="#3498db", label="Treatment"),
    mpatches.Patch(facecolor="#e74c3c", label="Outcome"),
    mpatches.Patch(facecolor="#95a5a6", label="Shared confounder"),
    mpatches.Patch(facecolor="#2ecc71", label="Regular-only treatment"),
    mpatches.Patch(facecolor="#f39c12", label="Aggregator-only confounder"),
]
fig.legend(
    handles=legend_elements, loc="upper center", ncol=5,
    fontsize=10, framealpha=0.9, bbox_to_anchor=(0.5, 0.98),
)

plt.tight_layout(rect=[0, 0.06, 1, 0.93])
plt.show()

## 6. Testable Implications

A good DAG makes **testable predictions**. If our DAG is correct, then certain conditional independencies should hold in the data. If they don't, the DAG is wrong — or at least incomplete.

For example, our DAG implies:
1. **`enrollment_month` is independent of `retention_action_taken` given the other confounders** — because enrollment_month has no direct edge to treatment (it only affects churn directly). This is a strong claim: after controlling for visit frequency, tenure, location churn rate, and payment delays, the month someone enrolled should not predict whether they received a retention action.
2. **`plan_price` is independent of `retention_action_taken` given the confounders** — for the same reason: plan price affects churn but is not used by managers in treatment assignment decisions.
3. **`visit_frequency` is NOT independent of `churned_30d` given treatment** — because visit_frequency has a direct effect on churn beyond the treatment path.

These are falsifiable claims. If the data contradicts them, we need to revise the DAG before estimating any causal effects. This is fundamentally different from ML model validation — we're validating our **assumptions**, not our predictions.

In [ ]:
# ── Test conditional independence implications ────────────────────────────────
# We use partial correlation as a proxy for conditional independence.
# Limitation: partial correlation only detects LINEAR dependencies. A true
# CI test would use kernel-based methods (e.g., KCIT), but those are
# computationally expensive at our scale. Partial correlation is a useful
# first-pass filter.

from scipy import stats

def partial_correlation(
    data: pd.DataFrame,
    x: str,
    y: str,
    covariates: list[str],
) -> dict:
    """Compute partial correlation between x and y controlling for covariates.

    Uses OLS residualization: regress x on covariates, regress y on covariates,
    then correlate the residuals. This removes the linear influence of covariates.
    """
    from numpy.linalg import lstsq

    df = data[[x, y] + covariates].dropna()
    Z = df[covariates].values
    Z = np.column_stack([Z, np.ones(len(Z))])  # add intercept

    # Residualize x
    x_vals = df[x].values
    beta_x, _, _, _ = lstsq(Z, x_vals, rcond=None)
    resid_x = x_vals - Z @ beta_x

    # Residualize y
    y_vals = df[y].values
    beta_y, _, _, _ = lstsq(Z, y_vals, rcond=None)
    resid_y = y_vals - Z @ beta_y

    # Correlate residuals
    r, p_value = stats.pearsonr(resid_x, resid_y)
    return {"partial_r": r, "p_value": p_value, "n": len(df)}


# Use the sample for faster computation
test_data = members_sample.copy()

# Conditioning set: confounders that connect to treatment
conditioning_set = ["visit_frequency", "tenure_months", "location_churn_rate", "payment_delay_flag"]

print("=" * 90)
print("TESTABLE IMPLICATIONS OF THE CAUSAL DAG")
print("=" * 90)

# Test 1: enrollment_month _||_ retention_action_taken | confounders
# Our DAG says enrollment_month does NOT directly cause treatment assignment
test1 = partial_correlation(
    test_data, "enrollment_month", "retention_action_taken", conditioning_set
)
print(f"\nTest 1: enrollment_month _||_ retention_action_taken | confounders")
print(f"  DAG prediction: independent (no direct edge to treatment)")
print(f"  Partial r = {test1['partial_r']:.4f}, p = {test1['p_value']:.4f}, n = {test1['n']:,}")
if test1["p_value"] > 0.05:
    print(f"  RESULT: CONSISTENT with DAG (p > 0.05, fail to reject independence)")
else:
    print(f"  RESULT: INCONSISTENT with DAG (p <= 0.05, may need to add edge)")
    print(f"  NOTE: Consider adding enrollment_month -> retention_action_taken edge")

# Test 2: plan_price _||_ retention_action_taken | confounders
test2 = partial_correlation(
    test_data, "plan_price", "retention_action_taken", conditioning_set
)
print(f"\nTest 2: plan_price _||_ retention_action_taken | confounders")
print(f"  DAG prediction: independent (managers don't use price in targeting)")
print(f"  Partial r = {test2['partial_r']:.4f}, p = {test2['p_value']:.4f}, n = {test2['n']:,}")
if test2["p_value"] > 0.05:
    print(f"  RESULT: CONSISTENT with DAG (p > 0.05, fail to reject independence)")
else:
    print(f"  RESULT: INCONSISTENT — consider adding plan_price -> retention_action_taken")

# Test 3: visit_frequency NOT _||_ churned_30d | retention_action_taken
# Our DAG says visit_frequency has a DIRECT effect on churn (not only through treatment)
test3 = partial_correlation(
    test_data, "visit_frequency", "churned_30d", ["retention_action_taken"]
)
print(f"\nTest 3: visit_frequency NOT _||_ churned_30d | treatment")
print(f"  DAG prediction: dependent (direct effect on churn beyond treatment path)")
print(f"  Partial r = {test3['partial_r']:.4f}, p = {test3['p_value']:.6f}, n = {test3['n']:,}")
if test3["p_value"] < 0.05:
    print(f"  RESULT: CONSISTENT with DAG (significant dependence, as expected)")
else:
    print(f"  RESULT: INCONSISTENT — visit_frequency should predict churn beyond treatment")

print("\n" + "=" * 90)
print("INTERPRETATION")
print("=" * 90)
print("""
If all tests are consistent: our DAG is at least not contradicted by the data.
This does NOT prove the DAG is correct — it only means we haven't found evidence
against it. Unfalsified ≠ true. But a DAG that fails its own testable implications
should not be used for causal estimation.

Limitation: These tests check linear conditional independence only. Non-linear
dependencies may be missed. For a more rigorous check, consider kernel-based
conditional independence tests (KCIT) or randomization-based tests.
""")

In [ ]:
# ── Visualize the confounding structure quantitatively ─────────────────────────
# Show how confounders relate to both treatment and outcome — the visual
# evidence that naive estimation is biased.

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle(
    "Confounding Evidence: Each Variable Correlates with Both Treatment and Outcome\n"
    "This is WHY naive comparison is biased — these create backdoor paths",
    fontsize=14, fontweight="bold", y=1.02,
)

confounder_list = [
    ("visit_frequency", "Visit Frequency (30d)"),
    ("tenure_months", "Tenure (months)"),
    ("plan_price", "Plan Price (R$)"),
    ("location_churn_rate", "Location Churn Rate"),
    ("enrollment_month", "Enrollment Month"),
    ("payment_delay_flag", "Payment Delay Flag"),
]

for idx, (var, label) in enumerate(confounder_list):
    ax = axes[idx // 3, idx % 3]

    # Group by treatment status and show distribution of confounder
    treated = test_data.loc[test_data["retention_action_taken"] == 1, var].dropna()
    control = test_data.loc[test_data["retention_action_taken"] == 0, var].dropna()

    if var == "payment_delay_flag":
        # Binary variable — bar chart
        x_pos = np.array([0, 1])
        width = 0.35
        ax.bar(
            x_pos - width / 2,
            [control.mean(), 1 - control.mean()],
            width, label="No action", color="#3498db", alpha=0.7,
        )
        ax.bar(
            x_pos + width / 2,
            [treated.mean(), 1 - treated.mean()],
            width, label="Action taken", color="#e74c3c", alpha=0.7,
        )
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Has delay", "No delay"])
        ax.set_ylabel("Proportion")
    elif var == "enrollment_month":
        # Categorical — grouped bar
        months_ctrl = control.value_counts().sort_index() / len(control)
        months_treat = treated.value_counts().sort_index() / len(treated)
        all_months = sorted(set(months_ctrl.index) | set(months_treat.index))
        x_pos = np.arange(len(all_months))
        width = 0.35
        ax.bar(
            x_pos - width / 2,
            [months_ctrl.get(m, 0) for m in all_months],
            width, label="No action", color="#3498db", alpha=0.7,
        )
        ax.bar(
            x_pos + width / 2,
            [months_treat.get(m, 0) for m in all_months],
            width, label="Action taken", color="#e74c3c", alpha=0.7,
        )
        ax.set_xticks(x_pos)
        ax.set_xticklabels(all_months, fontsize=8)
        ax.set_ylabel("Proportion")
    else:
        # Continuous — overlapping histograms
        bins = 30
        ax.hist(control, bins=bins, alpha=0.6, label="No action", color="#3498db", density=True)
        ax.hist(treated, bins=bins, alpha=0.6, label="Action taken", color="#e74c3c", density=True)
        ax.set_ylabel("Density")

    ax.set_title(label, fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 7. Key Takeaways

1. **The DAG is the foundation** — without it, causal claims are unjustified. Every edge and every missing edge is a substantive assumption that can be debated and tested.

2. **Adjustment set identified:** `visit_frequency`, `tenure_months`, `plan_price`, `location_churn_rate`, `enrollment_month`, `payment_delay_flag`. These are the variables we must condition on to close all backdoor paths between `retention_action_taken` and `churned_30d`.

3. **Separate DAGs for Regular vs Aggregator are causally necessary**, not just statistically convenient. The causal structure differs because the gym's intervention set differs. This is documented in ADR-003 and enforced in `configs/causal.yaml`.

4. **Naive comparison is directionally wrong.** The raw data shows treated members churn MORE than untreated — but this is entirely due to selection bias (managers target at-risk members). The causal DAG tells us what adjustments are needed to recover the true effect.

5. **Unmeasured confounding remains a risk.** Variables like "member motivation" and "life circumstances" are real but unobservable. We address this with sensitivity analysis in notebook 04 (Rosenbaum bounds), but we can never fully eliminate this concern from observational data.

6. **Testable implications provide a sanity check** but do not validate the DAG. Passing conditional independence tests means the DAG is consistent with the data — not that it's correct. The ultimate test is whether the causal estimates yield actionable and economically sensible results when deployed (notebook 05).

---

**Next:** [04_treatment_effects.ipynb](./04_treatment_effects.ipynb) — estimate the ATE and CATE using DoWhy and EconML's CausalForestDML, with refutation tests and sensitivity analysis.